# Análisis predictivo del aterrizaje (clasificación)
Se comparan Regresión Logística, SVM, Árbol de Decisión y KNN con `GridSearchCV`; se evalúan con exactitud y matriz de confusión.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
def plot_cm(y,yp,t):
    cm=confusion_matrix(y,yp);ax=plt.subplot();sns.heatmap(cm,annot=True,ax=ax,fmt='d')
    ax.set_xlabel('Predicho');ax.set_ylabel('Real');ax.set_title(t)
    ax.xaxis.set_ticklabels(['No aterrizó','Aterrizó']);ax.yaxis.set_ticklabels(['No aterrizó','Aterrizó']);plt.show()
def load(n):
    try:return pd.read_csv(n)
    except FileNotFoundError:return pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/'+n)
data=load('dataset_part_2.csv');X=load('dataset_part_3.csv')
Y=data['Class'].to_numpy()

In [ ]:
X=preprocessing.StandardScaler().fit(X).transform(X)
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=2)
print(X_test.shape)

In [ ]:
params={'C':[0.01,0.1,1],'penalty':['l2'],'solver':['lbfgs']}
logreg_cv=GridSearchCV(LogisticRegression(),params,cv=10).fit(X_train,Y_train)
print(logreg_cv.best_params_,logreg_cv.best_score_,'test:',logreg_cv.score(X_test,Y_test))
plot_cm(Y_test,logreg_cv.predict(X_test),'Regresión Logística')

In [ ]:
params={'kernel':('linear','rbf','poly','sigmoid'),'C':np.logspace(-3,3,5),'gamma':np.logspace(-3,3,5)}
svm_cv=GridSearchCV(SVC(),params,cv=10).fit(X_train,Y_train)
print(svm_cv.best_params_,svm_cv.best_score_,'test:',svm_cv.score(X_test,Y_test))
plot_cm(Y_test,svm_cv.predict(X_test),'SVM')

In [ ]:
params={'criterion':['gini','entropy'],'splitter':['best','random'],'max_depth':[2*n for n in range(1,10)],
'max_features':['sqrt','log2'],'min_samples_leaf':[1,2,4],'min_samples_split':[2,5,10]}
tree_cv=GridSearchCV(DecisionTreeClassifier(),params,cv=10).fit(X_train,Y_train)
print(tree_cv.best_params_,tree_cv.best_score_,'test:',tree_cv.score(X_test,Y_test))
plot_cm(Y_test,tree_cv.predict(X_test),'Árbol de Decisión')

In [ ]:
params={'n_neighbors':list(range(1,11)),'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]}
knn_cv=GridSearchCV(KNeighborsClassifier(),params,cv=10).fit(X_train,Y_train)
print(knn_cv.best_params_,knn_cv.best_score_,'test:',knn_cv.score(X_test,Y_test))
plot_cm(Y_test,knn_cv.predict(X_test),'KNN')

In [ ]:
res=pd.DataFrame({'Modelo':['Regresión Logística','SVM','Árbol de Decisión','KNN'],
'CV (mejor)':[m.best_score_ for m in (logreg_cv,svm_cv,tree_cv,knn_cv)],
'Prueba':[m.score(X_test,Y_test) for m in (logreg_cv,svm_cv,tree_cv,knn_cv)]})
print(res);res.set_index('Modelo').plot.bar();plt.ylim(0,1);plt.show()
print('Mejor modelo en validación cruzada:',res.loc[res['CV (mejor)'].idxmax(),'Modelo'])

## Conclusiones
- El mejor modelo se elige por la exactitud en validación cruzada; con un conjunto de prueba de solo 18 registros, varios modelos empatan.
- El error principal son los falsos positivos (predice aterrizaje cuando falló).
- Ideas: añadir clima y viento, probar Random Forest/XGBoost y calibrar probabilidades para estimar costos.